# Reinforcement Learning

# 3. Online evaluation

This notebook presents the online evaluation of a policy by **Monte-Carlo learning** and **TD learning**.

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors.
* Clear all cell outputs before upload on Moodle.

In [ ]:
import numpy as np

In [ ]:
from model import Maze, Walk, TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineEvaluation
from dynamic import PolicyEvaluation

import matplotlib.pyplot as plt
from scipy.stats import spearmanr

## To do

* Complete the method ``train`` of the class ``MCLearning`` below.
* Test Monte-Carlo learning on the random walk after 1 vs 100 episodes, each for a time horizon $t=100$.<br>
Interpret the results.
* Compare with the exact solution obtained by Dynamic Programming.<br> You might plot the [Spearman's correlation](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient) between both value functions with respect to the number of episodes. 

## Monte-Carlo learning

In [ ]:
class MCLearning(OnlineEvaluation):
    """Online evaluation by Monte-Carlo."""
        
    def train(self, state=None, horizon=100):
        """Train the model on one episode.
        
        Parameters
        ----------
        state : 
            First state of the episode (default = initial state of the environment).
        horizon : int
            Time horizon.
        """
        stop, states, rewards = self.get_episode(state=state, horizon=horizon)
        # remove last state
        states.pop()
        return_ = 0
        # backward update
        for state, reward in zip(reversed(states), reversed(rewards)):
            self.add_state(state)
            code = self.environment.encode(state)
            self.count[code] += 1
            # to be modified
            # begin
            return_ = reward + self.gamma * return_
            # end 
            diff = return_ - self.value[code]
            count = self.count[code]
            self.value[code] += diff / count
            

## Walk

In [ ]:
walk = Walk()

In [ ]:
algo = MCLearning(walk, policy='random', gamma=0.9)

In [ ]:
algo.train()

In [ ]:
values = algo.get_values()

In [ ]:
walk.display_values(values)

In [ ]:
policy = algo.improve_policy()

In [ ]:
walk.display_policy(policy)

In [ ]:
walk = Walk()
algo = MCLearning(walk, policy='random', gamma=0.9)

for i in range(100):
    algo.train()
    
values = algo.get_values()
walk.display_values(values)
policy = algo.improve_policy()
walk.display_policy(policy)

ANSWER:

After around 100 Monte-Carlo episodes, the estimated value function becomes much more structured, allowing the improved policy to consistently direct actions toward higher-value regions. Although still slightly noisy, MC already yields a significantly better policy than the initial random one.

In [ ]:
dp = PolicyEvaluation(walk, policy='random', gamma=0.9)
dp.evaluate_policy()
values_dp = dp.values

walk.display_values(values_dp)
policy = algo.improve_policy()
walk.display_policy(policy)

ANSWER:

Monte-Carlo values are initially noisy but gradually converge toward the smooth and accurate value function computed by Dynamic Programming. The resulting greedy policy becomes closer to the optimal policy derived from the DP values.

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import numpy as np

walk = Walk()

dp = PolicyEvaluation(walk, policy='random', gamma=0.9)
dp.evaluate_policy()
values_dp = np.array(dp.values) 

episodes_list = [1, 5, 10, 20, 50, 100, 200, 500]
spearman_scores = []

for n in episodes_list:

    walk = Walk()
    algo = MCLearning(walk, policy='random', gamma=0.9)

    for _ in range(n):
        algo.train()

    values_mc = np.array(algo.get_values()) 

    rho = spearmanr(values_mc, values_dp).correlation
    spearman_scores.append(rho)

    print("Episodes :", n, "  Spearman =", rho)

plt.plot(episodes_list, spearman_scores, marker='o')
plt.xlabel("Nombre d'épisodes Monte-Carlo")
plt.ylabel("Corrélation Spearman (MC vs DP)")
plt.title("Convergence MC vers DP")
plt.grid(True)
plt.show()

## To do

* Complete the method ``train`` of the class ``TDLearning``.
* Do the same experiments as above on the Walk.
* Which algorithm learns faster on this environment?


## TD learning

In [ ]:
class TDLearning(OnlineEvaluation):
    """Online evaluation by TD learning."""
        
    def train(self, state=None, horizon=100):
        """Train the model on one episode.
        
        Parameters
        ----------
        state : 
            First state of the episode (default = initial state of the environment).
        horizon : int
            Time horizon.
        """
        self.environment.reset(state)
        for t in range(horizon):
            state = self.environment.state
            self.add_state(state)
            code = self.environment.encode(state)

            action = self.get_action(state)
            if action is None:
                break

            reward, stop = self.environment.step(action)
            next_state = self.environment.state
            next_code = self.environment.encode(next_state)
            td_target = reward + self.gamma * self.value[next_code]
            td_error = td_target - self.value[code]

            self.count[code] += 1
            alpha = 1 / self.count[code]
            self.value[code] += alpha * td_error

            if self.environment.is_terminal(next_state):
                break

In [ ]:
walk = Walk()
algo = TDLearning(walk, policy='random', gamma=0.9)
algo.train()
values = algo.get_values()
walk.display_values(values)
policy = algo.improve_policy()
walk.display_policy(policy)

In [ ]:
walk = Walk()
algo = TDLearning(walk, policy='random', gamma=0.9)

for i in range(100):
    algo.train()
    
values = algo.get_values()
walk.display_values(values)
policy = algo.improve_policy()
walk.display_policy(policy)

In [ ]:
dp = PolicyEvaluation(walk, policy='random', gamma=0.9)
dp.evaluate_policy()
values_dp = dp.values

walk.display_values(values_dp)
policy = algo.improve_policy()
walk.display_policy(policy)

ANSWER:

The value function and policy learned by TD closely match the exact solution obtained by dynamic programming, confirming that TD converges toward the true optimal values.

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import numpy as np

walk = Walk()

dp = PolicyEvaluation(walk, policy='random', gamma=0.9)
dp.evaluate_policy()
values_dp = np.array(dp.values) 

episodes_list = [1, 5, 10, 20, 50, 100, 200, 500]
spearman_scores = []

for n in episodes_list:

    walk = Walk()
    algo = TDLearning(walk, policy='random', gamma=0.9)

    for _ in range(n):
        algo.train()

    values_mc = np.array(algo.get_values()) 

    rho = spearmanr(values_mc, values_dp).correlation
    spearman_scores.append(rho)

    print("Episodes :", n, "  Spearman =", rho)

plt.plot(episodes_list, spearman_scores, marker='o')
plt.xlabel("Nombre d'épisodes TD Learning")
plt.ylabel("Corrélation Spearman (TD vs DP)")
plt.title("Convergence TD vers DP")
plt.grid(True)
plt.show()

ANSWER:

TD learning rapidly converges toward the dynamic programming solution. It reaches above 0.95 Spearman correlation after only a few episodes.
Finally, TD learning learns faster than MC Learning.

## To do

Test the other environments:
* The maze: can you find the exit after policy improvement?<br> You might adapt the number of episodes used for training.
* The games (Tic-Tac-Toe, Nim, Connect Four): can you beat a random player after policy improvement? an adversary with the one-step policy?<br> Comment the results.

## Maze

In [ ]:
maze_map = np.load('maze.npy')

In [ ]:
init_state = (1, 0)
exit_state = (1, 20)
Maze.set_parameters(maze_map, init_state, [exit_state])

In [ ]:
maze = Maze()

In [ ]:
maze.display()

In [ ]:
algo = MCLearning(maze, policy='random')

In [ ]:
n_episodes = 10000
for t in range(n_episodes):
    algo.train(state='random')

In [ ]:
values = algo.get_values()
maze.display_values(values)

In [ ]:
policy = algo.improve_policy()
maze.display_policy(policy)

ANSWER:

The agent didn't find the exit after policy improvement, even with a large number of episodes. The reward is sparse and the exit is rarely visited, so the improved policy does not guide the agent toward the goal.

## Games

In [ ]:
Game = TicTacToe

In [ ]:
# random player
game = Game()
agent = Agent(game)

In [ ]:
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
algo = MCLearning(game, policy='random', gamma=0.9)

n_episodes = 10000
for _ in range(n_episodes):
    algo.train()

policy = algo.improve_policy()

In [ ]:
agent_improved = Agent(game, policy=policy)

returns = agent_improved.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

ANSWER:

After policy improvement, the agent performs extremely well. It easily beats a random player, winning almost all games.

In [ ]:
agent_vs_one_step = Agent(game, policy=policy)

game = Game(adversary_policy='one_step')

returns = agent_vs_one_step.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

ANSWER:

Against a stronger opponent using the one-step policy, the agent still wins the large majority of matches, showing that it has learned a strategy that is close to optimal for Tic-Tac-Toe.
Overall, policy improvement allows the agent to reach a level of play that is superior to both weak and stronger opponents.

In [ ]:
Game = Nim
game = Game()
agent = Agent(game)
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
algo = MCLearning(game, policy='random', gamma=0.9)

n_episodes = 10000
for _ in range(n_episodes):
    algo.train()

policy = algo.improve_policy()

In [ ]:
agent_improved = Agent(game, policy=policy)

returns = agent_improved.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

ANSWER:

The improved agent wins almost all games against a random oppponent.

In [ ]:
agent_vs_one_step = Agent(game, policy=policy)

game = Game(adversary_policy='one_step')

returns = agent_vs_one_step.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

ANSWER:

Against a stronger opponent using the one-step policy, the agent still wins the large majority of matches. This shows that MC learning successfully discovers a strong winning strategy. However, the agent performs well in Nim, but not as strongly as in Tic-Tac-Toe.

In [ ]:
Game = ConnectFour
game = Game()
agent = Agent(game)
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
algo = MCLearning(game, policy='random', gamma=0.9)

n_episodes = 10000
for _ in range(n_episodes):
    algo.train()

policy = algo.improve_policy()

In [ ]:
agent_improved = Agent(game, policy=policy)

returns = agent_improved.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

In [ ]:
agent_vs_one_step = Agent(game, policy=policy)

game = Game(adversary_policy='one_step')

returns = agent_vs_one_step.get_returns(n_episodes=1000)
print(np.unique(returns, return_counts=True))

ANSWER:

In Connect Four, Monte-Carlo learning produces a significantly stronger policy than the random player but still struggles against the one-step adversary. This is expected, as the game is more complex than Tic-Tac-Toe or Nim, making it harder for MC learning to discover consistently winning strategies.